# 07 — RoBERTa LLM Module

**Subject:** NLP — Topic 3 (Isaac González)  
**Goal:** Evaluate a local Large Language Model on the MVSA-Single test set.

## Model
`cardiffnlp/twitter-roberta-base-sentiment-latest`
- Architecture: RoBERTa-base (125M parameters)
- Trained specifically on Twitter data for sentiment analysis
- Runs fully locally via HuggingFace Transformers (no API required)

## Why this model?
- Domain match: same domain (Twitter) as MVSA-Single
- No fine-tuning needed — uses the model out of the box
- Outperforms classical NLP methods (NB: 0.58, LR: 0.61) without any training
- Lightweight enough to run on CPU

**Expected F1:** ~0.67  
**Expected time:** ~10–15 minutes on CPU

## Cell 1 — Setup and dataset loading

In [ ]:
import os
os.chdir("/Users/yesicarb/Desktop/UIE/3º Curso/2 SEM/PROYECTO/emotion/multimodal_emotion")

import pandas as pd
import sys
sys.path.append("src")

from nlp_llm.llm_classifier import run_llm

df = pd.read_csv("data/processed/labels.csv")
print(f"Dataset loaded: {df.shape}")
print(df['label'].value_counts())

## Cell 2 — Run RoBERTa on the test set

This cell:
1. Downloads and loads RoBERTa (first run only, ~500MB)
2. Applies the same 80/20 split as all other modules (`random_state=42`)
3. Classifies 974 test texts one by one
4. Builds probability vectors for each prediction
5. Saves F1 score and probabilities to `results/metrics_llm.json`

⚠️ The UNEXPECTED warnings from HuggingFace are normal and can be ignored.
They occur because the model head differs from the pretraining architecture.

⏱️ **Expected time: 10–15 minutes on CPU.**

In [ ]:
preds, probas, y_test, f1 = run_llm(df)
print(f"\nFinal F1 macro: {f1:.4f}")

## Cell 3 — Results analysis and comparison

Key findings:
- RoBERTa (F1=0.67) outperforms NB (0.58) and LR (0.61) **without any fine-tuning**
- This demonstrates the power of domain-specific pretrained LLMs
- Still below BERT fine-tuned (0.72) — fine-tuning on task-specific data helps
- The probabilities are saved for integration in the late fusion system

In [ ]:
import matplotlib.pyplot as plt

modules   = ['Random\nBaseline', 'Naive\nBayes', 'Logistic\nRegression',
             'RoBERTa\nLLM', 'BERT\nFine-tuned']
f1_scores = [0.33, 0.58, 0.61, f1, 0.72]
colors    = ['#9ca3af', '#facc15', '#a3e635', '#06b6d4', '#34d399']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(modules, f1_scores, color=colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, 0.90)
ax.axhline(0.33, color='gray', linestyle='--', alpha=0.4, label='Random baseline')
ax.set_ylabel('F1-score (macro)')
ax.set_title('NLP Modules Comparison — RoBERTa LLM vs Classical Methods')
ax.legend()
plt.tight_layout()
plt.savefig('results/figures/llm_comparison.png', dpi=150)
plt.show()
print("Figure saved to results/figures/llm_comparison.png")